# HACK AI / Intro to Large Language Modelling
## BERT for supervised multi-class classification: load our previously fine-tuned model

### *Mariam Cook*

### *m.cook6@exeter.ac.uk*

### *University of Exeter Centre for Computational Social Science*



Code credit: Claude Code Sonnet 4.6

In [ ]:
import torch
from transformers import AutoModel
import torch.nn as nn
import numpy as np

In [ ]:
# ================================================
# 1. Check GPU is available
# ================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")  # sanity check — should print 'cuda'


In [ ]:
# ================================================
# 2. Define the same model architecture as before
# ================================================
class BertClassifier(nn.Module):
    def __init__(self, num_classes):
        super(BertClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained('bert-base-cased')
        self.dense = nn.Linear(768, 1024)
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.relu(self.dense(pooled_output))
        return self.classifier(x)


In [ ]:
# Mount your google drive

from google.colab import drive
import os

# ✅ Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# ================================================
# 3. Load checkpoint onto GPU
# ================================================
checkpoint = torch.load(
    '/content/drive/MyDrive/....../party_model.pt', # this needs to be a link to the model that you saved in YOUR Google drive
    map_location=device  # maps directly to cuda if available
)

In [ ]:
# ================================================
# 4. Reinstantiate model, load weights and move to GPU
# ================================================
model = BertClassifier(num_classes=checkpoint['num_classes'])
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)  # move model to GPU
model.eval()

# ================================================
# 5. Recover label mapping
# ================================================
class_label_dict = checkpoint['class_label_dict']

print("Model loaded successfully on:", device)

## Run Inference

### Let's see results for previously unseen text

#### Function to take some text, split and encode using BERT tokenizer

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

def prep_data(text):
    tokens = tokenizer(text, max_length=512, truncation=True, padding='max_length',
                       add_special_tokens=True, return_token_type_ids=False,
                       return_tensors='pt')
    return {
        'input_ids': tokens['input_ids'].long(),
        'attention_mask': tokens['attention_mask'].long()
    }

#### Function to use the trained BERT model to predict the class of some new text

In [ ]:
def get_model_prediction(input_text):
    tokenized_text = prep_data(input_text)

    model.eval()
    with torch.no_grad():
        input_ids = tokenized_text['input_ids'].to(device)
        attention_mask = tokenized_text['attention_mask'].to(device)
        logits = model(input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]  # convert to probabilities

    top_class = list(class_label_dict.keys())[list(class_label_dict.values()).index(np.argmax(probs))]
    confidence_level = np.amax(probs)
    print(top_class)
    return top_class, confidence_level

We looked at these two new examples before, check the results are the same:
https://www.theyworkforyou.com/debates/?id=2023-06-27b.170.2

In [ ]:
text_to_predict = 'I beg to move, That this House is extremely concerned that, under this Conservative Government, average \
                  mortgage costs will be increasing by £2,900 per year, with a typical household in the UK paying over £2,000 more per year \
                  than in France and over £1,000 more than in Ireland and Belgium, and that renters face huge increases in rent payments;.'
get_model_prediction(text_to_predict)

In [ ]:
text_to_predict = 'I am here to account for what has happened in the UK. Obviously, there are differences—[Interruption.] If I may answer. \
                  There are differences across the EU and the US. What I am telling the House, which is quite transparently clear, \
                  is that inflationary pressures are affecting all economies at the moment, and it is my responsibility to account for what we are doing as a Government. \
                  holocaust memorial and education centre. I understand that the Standing Orders Committee has considered \
                  the progress of the Holocaust Memorial Bill, which will bring both the much-needed and expected education centre \
                  and the memorial to fruition. Can my right hon. Friend provide a progress report on that Bill, but also on the \
                  long-promised boycotts, divestment and sanctions Bill that the Government have promised to bring forward?'
get_model_prediction(text_to_predict)

## Continue training

If you have more examples of labelled text you could keep training on top of the fine-tuning done previously, rather than train from scratch.

ADVICE FROM CLAUDE CODE: The key things to be aware of:





*   Optimizer state matters — the optimizer_state_dict saves Adam's internal momentum and variance estimates for every parameter, meaning training resumes smoothly from where it left off rather than restarting Adam's adaptive learning rate from scratch. This is why saving it was good practice.
*   New examples should match the same format — the new training data needs to go through the same tokenization pipeline (prep_data, same seq_len=512) and use the same class_label_dict mapping so labels are consistent.
* Learning rate — you may want to use a lower learning rate than the original (e.g. lr=1e-6) when fine-tuning further to avoid overwriting what the model has already learned, especially if the new dataset is small.
* Catastrophic forgetting — if the new examples are from a very different distribution (e.g. social media text vs parliamentary speech), continued training could degrade performance on the original data. If that's a concern, mixing some original training examples in with the new ones helps mitigate this.
